# Prism-T GPU High-Concurrency Visual Manga Translation (Google Colab)

Deploy a local OpenAI-compatible Vision-Language Model (**vLLM**) alongside **Prism-T** (`prismtd` + CUDA `prism-t-engine`) on Google Colab GPU (**T4 15GB / L4 24GB / A100 40GB+**).

### Why This Architecture Hits Maximum Speed + Concurrency + Quality
- **Continuous Batching + Prefix Caching + Guided JSON**: `vLLM` batches concurrent pages automatically, caches Prism-T's repeated system prompt/glossary prefix, and enforces strict OpenAI `json_schema` decoding (zero malformed JSON retries).
- **Zero Tunnel / Zero Ban Risk (`all_in_one` mode)**: `vLLM` and `Prism-T` communicate over `127.0.0.1` inside the same Colab VM — zero public tunnel risk and zero image upload latency.
- **Crash-Safe Google Drive Persistence**: Input `.zip`/`.cbz` is unpacked once to local NVMe SSD (`/content/work_in`) for fast reads; SQLite WAL runs on local SSD and hot-backups to Drive every 60s; translated PNGs stream straight to Drive with `--resume` across Colab disconnects.

### Verified Models (Speed × Concurrency × Quality)
| Model | T4 15GB VRAM | Throughput / Concurrency | Notes |
| :--- | :--- | :--- | :--- |
| `Qwen/Qwen2.5-VL-7B-Instruct-AWQ` | ~5.8 GB | **6–8 concurrent** (~2.2–3.5s/page) | **Best all-around default on T4**: top JP vertical OCR + natural ZH prose + strict JSON. |
| `huihui-ai/Qwen2.5-VL-7B-Instruct-abliterated` | ~6.2 GB (4-bit BnB on T4) / 15 GB (BF16 on L4) | **4–8 concurrent** | **Zero-refusal / uncensored** variant for mature manga (auto-loads 4-bit bitsandbytes on T4, native BF16 on L4/A100). |
| `cyankiwi/Qwen3-VL-8B-Instruct-AWQ-4bit` | ~6.0 GB | **6–8 concurrent** | Newer Qwen3-VL 8B 4-bit AWQ; stronger literary prose (requires latest vLLM). |
| `Qwen/Qwen2.5-VL-3B-Instruct-AWQ` | ~2.8 GB | **12–16 concurrent** (~1.2–2.0s/page) | Ultra-fast high-concurrency fallback to finish ~2,000 pages in ~1 hour on free T4. |

> **Setup before clicking Run All**:
> 1. In Colab menu select **Runtime -> Change runtime type -> T4 GPU** (or **L4 / A100** if on Colab Pro — L4 finishes 1,910 pages in ~45–60 min).
> 2. Zip your raw PNG folder with **Store / No Compression** (takes ~5s) and upload to Google Drive (e.g. `MyDrive/prism_t_jobs/manga.zip`).

In [ ]:
#@title 1. Configure Job & Auto-Detect GPU
import os, torch

MODE = "all_in_one" #@param ["all_in_one", "vllm_only"]
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct-AWQ" #@param ["Qwen/Qwen2.5-VL-7B-Instruct-AWQ", "huihui-ai/Qwen2.5-VL-7B-Instruct-abliterated", "cyankiwi/Qwen3-VL-8B-Instruct-AWQ-4bit", "Qwen/Qwen2.5-VL-3B-Instruct-AWQ"] {allow-input: true}
DRIVE_INPUT_PATH = "/content/drive/MyDrive/prism_t_jobs/manga.zip" #@param {type:"string"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/prism_t_jobs/output_zh" #@param {type:"string"}
WORK_TITLE = "MangaWork" #@param {type:"string"}
CHAPTER_TITLE = "Chapter 1" #@param {type:"string"}
TARGET_LANG = "zh-CN" #@param ["zh-CN", "zh-TW", "en", "ko"]
PRESERVE_ART_TEXT = True #@param {type:"boolean"}
CONCURRENCY_OVERRIDE = 0 #@param {type:"integer"}
MAX_PIXELS = 602112 #@param {type:"integer"}
PRISM_T_REPO = "https://github.com/senran-N/prism-t.git" #@param {type:"string"}

assert torch.cuda.is_available(), "No CUDA GPU detected! Go to Runtime -> Change runtime type -> GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
cpu_cnt = os.cpu_count() or 2

is_t4 = (cc_major == 7 and cc_minor == 5) or (vram_gb < 18.0)
is_awq = ("awq" in MODEL_ID.lower())
is_3b = ("3b" in MODEL_ID.lower())

if MODE == "vllm_only":
    GPU_MEM_UTIL = 0.90
    AUTO_CONC = 12 if (vram_gb >= 22 or is_3b) else 8
elif is_t4:
    # Reserve ~3.5-4.0 GB VRAM for Prism-T CTD ONNX + LaMa on 15 GB T4
    GPU_MEM_UTIL = 0.68 if not is_3b else 0.72
    AUTO_CONC = 10 if is_3b else 6
elif vram_gb >= 38:
    GPU_MEM_UTIL = 0.82
    AUTO_CONC = 20
else:
    # L4 24 GB / 3090 / 4090
    GPU_MEM_UTIL = 0.78
    AUTO_CONC = 12

CONCURRENCY = CONCURRENCY_OVERRIDE if CONCURRENCY_OVERRIDE > 0 else AUTO_CONC
VLLM_DTYPE = "half" if (is_t4 or is_awq) else "bfloat16"

if is_awq:
    VLLM_QUANT = "awq_marlin" if cc_major >= 8 else "awq"
elif vram_gb < 20.0:
    # Unquantized 7B/8B model on 15 GB T4 -> 4-bit bitsandbytes online quant
    VLLM_QUANT = "bitsandbytes"
else:
    VLLM_QUANT = ""

print(f"GPU: {gpu_name} ({vram_gb:.1f} GB, sm_{cc_major}{cc_minor}) | CPUs: {cpu_cnt}")
print(f"Mode: {MODE} | Model: {MODEL_ID}")
print(f"vLLM: dtype={VLLM_DTYPE}, quant={VLLM_QUANT or 'none'}, gpu_util={GPU_MEM_UTIL}, max_pixels={MAX_PIXELS}")
print(f"Prism-T Concurrency: {CONCURRENCY} | PreserveArtText: {PRESERVE_ART_TEXT}")

In [ ]:
#@title 2. Mount Google Drive & Stage Input Archive to Local Fast SSD
import shutil, zipfile, time
from pathlib import Path

if MODE == "all_in_one":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    src = Path(DRIVE_INPUT_PATH)
    assert src.exists(), f"Input not found on Google Drive: {src}"

    out_dir = Path(DRIVE_OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    stage_dir = Path("/content/work_in")
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    if src.is_file() and src.suffix.lower() in (".zip", ".cbz"):
        print(f"Extracting {src.name} to local SSD {stage_dir} ...")
        with zipfile.ZipFile(src, "r") as zf:
            for m in zf.infolist():
                if m.is_dir():
                    continue
                fname = Path(m.filename).name
                if not fname or fname.startswith("."):
                    continue
                with zf.open(m) as rf, open(stage_dir / fname, "wb") as wf:
                    shutil.copyfileobj(rf, wf)
        print(f"Extracted {len(list(stage_dir.iterdir()))} files in {time.time()-t0:.1f}s.")
    elif src.is_dir():
        print(f"Copying directory {src} to local SSD {stage_dir} ...")
        shutil.copytree(src, stage_dir, dirs_exist_ok=True)
        print(f"Copied {len(list(stage_dir.iterdir()))} files in {time.time()-t0:.1f}s.")
    else:
        raise ValueError(f"Unsupported DRIVE_INPUT_PATH: {src}")

    # Keep SQLite WAL on local SSD (Google Drive FUSE breaks SQLite WAL locks)
    DRIVE_DB_PATH = out_dir / "prism_t.db"
    LOCAL_DB_PATH = Path("/content/prism_t.db")
    if DRIVE_DB_PATH.is_file():
        shutil.copy2(DRIVE_DB_PATH, LOCAL_DB_PATH)
        print(f"Restored existing DB checkpoint ({LOCAL_DB_PATH.stat().st_size / 1e6:.1f} MB) for --resume.")
    else:
        print("Starting fresh SQLite DB on local SSD (auto-backed up to Drive every 60s).")
else:
    print("MODE=vllm_only: Skipping Google Drive staging.")

In [ ]:
#@title 3. Install Isolated vLLM Venv + Prism-T (Go + CUDA Engine + CJK Fonts)
import os, subprocess, shutil
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    res = subprocess.run(cmd, shell=True)
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed ({res.returncode}): {cmd}")
    return res.returncode

if not shutil.which("uv"):
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"{Path.home()}/.local/bin:/usr/local/go/bin:" + os.environ["PATH"]

if MODE == "all_in_one":
    sh("apt-get update -qq && apt-get install -y -qq fonts-noto-cjk sqlite3 > /dev/null")

# Isolated vLLM virtualenv so vLLM torch/transformers pins never clash with prism-t-engine
VLLM_VENV = Path("/content/vllm-venv")
if not (VLLM_VENV / "bin" / "vllm").exists():
    sh(f"uv venv {VLLM_VENV} --python 3.11")
    extra_pkgs = "bitsandbytes>=0.45.0" if VLLM_QUANT == "bitsandbytes" else ""
    sh(f"uv pip install --python {VLLM_VENV}/bin/python -q --upgrade vllm transformers accelerate qwen-vl-utils pillow {extra_pkgs}")
else:
    print("vLLM venv already installed.")

REPO_DIR = Path("/content/prism-t")
if MODE == "all_in_one":
    if not shutil.which("go"):
        sh("curl -fsSL https://go.dev/dl/go1.24.2.linux-amd64.tar.gz | tar -C /usr/local -xz")
    if not REPO_DIR.exists():
        sh(f"git clone --depth 1 {PRISM_T_REPO} {REPO_DIR}")
    else:
        sh(f"git -C {REPO_DIR} pull --ff-only")

    sh(f"cd {REPO_DIR} && CGO_ENABLED=0 go build -o prismtd ./cmd/prismtd")
    sh(f"uv --directory {REPO_DIR}/engine sync")

    # Verify engine venv CUDA works with Colab's driver; fall back to cu124 if driver < CUDA 12.8
    rc = sh(
        f"uv --directory {REPO_DIR}/engine run python -c 'import torch; assert torch.cuda.is_available()'",
        check=False,
    )
    if rc != 0:
        print("Colab CUDA driver < 12.8 -> installing PyTorch cu124 in engine/.venv ...")
        sh(f"uv pip install --python {REPO_DIR}/engine/.venv/bin/python --index-url https://download.pytorch.org/whl/cu124 --reinstall torch")
        sh(f"uv --directory {REPO_DIR}/engine run python -c 'import torch; print(torch.cuda.get_device_name(0))'")
    else:
        print("Engine CUDA verified OK!")

In [ ]:
#@title 4. Launch High-Concurrency vLLM Server & Run Guided JSON Schema Smoke Test
import json, subprocess, time, urllib.request
from pathlib import Path

subprocess.run("pkill -f 'vllm.entrypoints.openai.api_server' || true", shell=True)
time.sleep(1)

vllm_cmd = [
    "/content/vllm-venv/bin/python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--trust-remote-code",
    "--dtype", VLLM_DTYPE,
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--max-model-len", "8192",
    "--max-num-seqs", str(max(CONCURRENCY, 8)),
    "--enable-prefix-caching",
    "--limit-mm-per-prompt", json.dumps({"image": 2}),
    "--mm-processor-kwargs", json.dumps({"min_pixels": 200704, "max_pixels": int(MAX_PIXELS)}),
]
if VLLM_QUANT == "bitsandbytes":
    vllm_cmd += ["--quantization", "bitsandbytes", "--load-format", "bitsandbytes"]
elif VLLM_QUANT:
    vllm_cmd += ["--quantization", VLLM_QUANT]

if is_t4 and MODE == "all_in_one":
    # Saves ~1.2 GB CUDA graph overhead so CTD + LaMa coexist safely on 15 GB T4
    vllm_cmd += ["--enforce-eager"]

VLLM_LOG = Path("/content/vllm.log")
vllm_log_f = open(VLLM_LOG, "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log_f, stderr=subprocess.STDOUT)
print("Launching vLLM (first-time HuggingFace weight download takes 2-4 min)...")
print("Command:", " ".join(vllm_cmd))

ready = False
t0 = time.time()
for _ in range(360):
    if vllm_proc.poll() is not None:
        print(VLLM_LOG.read_text()[-4000:])
        raise RuntimeError(f"vLLM exited early with code {vllm_proc.returncode}")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    elapsed = int(time.time() - t0)
    if elapsed > 0 and elapsed % 20 == 0:
        print(f"  waiting for vLLM... ({elapsed}s)", flush=True)
    time.sleep(2)

assert ready, "vLLM timed out! Check /content/vllm.log:\n" + VLLM_LOG.read_text()[-3000:]
print(f"vLLM ready on http://127.0.0.1:8000/v1 in {time.time()-t0:.1f}s!")

probe_payload = json.dumps({
    "model": MODEL_ID,
    "temperature": 0.0,
    "messages": [{"role": "user", "content": "Reply with ok=true."}],
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "probe",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "required": ["ok"],
                "properties": {"ok": {"type": "boolean"}},
            },
        },
    },
}).encode()
req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=probe_payload,
    headers={"Content-Type": "application/json", "Authorization": "Bearer local-vllm"},
)
with urllib.request.urlopen(req, timeout=30) as resp:
    out = json.loads(resp.read().decode())
    print("Guided JSON Schema smoke test:", out["choices"][0]["message"]["content"])

In [ ]:
#@title 5. Run Prism-T High-Concurrency Translation (With 60s Drive Auto-Backup)
import os, shutil, sqlite3, subprocess, threading, time
from pathlib import Path

if MODE != "all_in_one":
    print("MODE=vllm_only: vLLM is listening on http://127.0.0.1:8000/v1.")
    print("Keep this runtime alive and point your Prism-T client to this OpenAI-compatible endpoint.")
else:
    stop_backup = threading.Event()
    DRIVE_DB_PATH = Path(DRIVE_OUTPUT_DIR) / "prism_t.db"
    LOCAL_DB_PATH = Path("/content/prism_t.db")

    def backup_db_once():
        if not LOCAL_DB_PATH.is_file():
            return
        tmp_dst = Path("/content/prism_t.db.bak")
        try:
            src_con = sqlite3.connect(str(LOCAL_DB_PATH), timeout=30.0)
            dst_con = sqlite3.connect(str(tmp_dst))
            src_con.backup(dst_con)
            dst_con.close()
            src_con.close()
            shutil.copy2(tmp_dst, DRIVE_DB_PATH)
        except Exception as e:
            print(f"[backup-warn] {e}", flush=True)

    def backup_loop():
        while not stop_backup.wait(60.0):
            backup_db_once()

    t_bak = threading.Thread(target=backup_loop, daemon=True)
    t_bak.start()

    env = os.environ.copy()
    env["PATH"] = f"{Path.home()}/.local/bin:/usr/local/go/bin:" + env["PATH"]
    env["PRISM_T_LLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
    env["PRISM_T_LLM_API_KEY"] = "local-vllm"
    env["PRISM_T_LLM_MODEL"] = MODEL_ID
    env["PRISM_T_LLM_VISION"] = "true"
    env["PRISM_T_SKIP_SOURCE_OCR"] = "true"
    env["PRISM_T_PRESERVE_ART_TEXT"] = "true" if PRESERVE_ART_TEXT else "false"
    env["PRISM_T_CONCURRENCY"] = str(CONCURRENCY)

    cmd = [
        "/content/prism-t/prismtd", "translate",
        "--in", "/content/work_in",
        "--out", DRIVE_OUTPUT_DIR,
        "--db", str(LOCAL_DB_PATH),
        "--engine-dir", "/content/prism-t/engine",
        "--work", WORK_TITLE,
        "--chapter", CHAPTER_TITLE,
        "--lang", TARGET_LANG,
        "--device", "cuda",
        "--stream",
        "--resume",
        "--continue-on-error",
    ]
    if PRESERVE_ART_TEXT:
        cmd.append("--preserve-art-text")

    print("Starting Prism-T translation:", " ".join(cmd), flush=True)
    t_start = time.time()
    try:
        proc = subprocess.Popen(
            cmd, cwd="/content/prism-t", env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        done_pages = 0
        for line in proc.stdout:
            print(line, end="", flush=True)
            if " done page #" in line:
                done_pages += 1
                if done_pages % 25 == 0:
                    dt = time.time() - t_start
                    rate = done_pages / max(dt, 1.0)
                    print(
                        f"=== [THROUGHPUT] {done_pages} pages in {dt/60:.1f}m | "
                        f"{1/rate:.2f} s/page ({rate*3600:.0f} pages/hr) ===",
                        flush=True,
                    )
        rc = proc.wait()
        print(f"Prism-T finished with exit code {rc} in {(time.time()-t_start)/60:.1f} min.")
    finally:
        stop_backup.set()
        backup_db_once()
        print(f"Final DB checkpoint saved to {DRIVE_DB_PATH}")